# Begin

In [1]:
# @launchit.collected

In [ ]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter # @launchit.collect
import json
import datetime
import pprint
import re
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp
import sqlite3

import lark # @launchit.collect

from tqdm.notebook import tqdm
from tqdm import tqdm as tqdm_text

import numpy as np
import einops
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as pltpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2.functional as tvtf

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
from dataset_utils import *
from models import *
from utils import * # @launchit.collect
from math_utils import *
from logging_utils import *
from image_utils import *
from model_registry import *
from torch_helpers import *
import launchit # @launchit.disable
from hp_utils import *
from autoincrement import Autoincrement
from basis_pursuit import *

# Init

In [ ]:
ArrayUtils.init()
LOG = Logging.get()
RNG = np.random.default_rng()

class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

CONFIG = namedtuple('Config', 
                    'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, mnist_path, private_data_path, run_path, ' + 
                    'self_fname, self_name, ' +
                    'subproject_name,' +
                    'is_cuda, cuda_device, exec_mode, is_interactive')(
    project_root_path=project_root_path,
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    subproject_path=os.path.abspath('.'),
    data_path=os.path.join(project_root_path, 'data'),
    mnist_path=os.path.join(project_root_path, 'data', 'mnist'),
    private_data_path=None,
    run_path=None,
    self_fname=None,
    self_name=None,
    subproject_name=None,
    is_cuda=torch.cuda.is_available(),
    cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
    exec_mode=ExecMode.MASTER_NOTEBOOK,
    is_interactive=True,
)

if IPython.get_ipython() is None:
    module_fname = __file__
    module_basename = os.path.basename(module_fname)
    module_name, _ = os.path.splitext(module_basename)
    
    CONFIG = CONFIG._replace(self_fname=module_fname, self_name=module_name)
    CONFIG = CONFIG._replace(exec_mode=ExecMode.LAUNCH_MODULE)
else:
    with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
        notebook_fname = json.load(cf)['jupyter_session']
        notebook_basename = os.path.basename(notebook_fname)
        notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    
        m = re.match(r'(\w+)-Copy\d+$', notebook_name)
    
        if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook

        CONFIG = CONFIG._replace(self_fname=notebook_fname, self_name=notebook_name)
        
        is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
        CONFIG = CONFIG._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)

CONFIG = CONFIG._replace(is_interactive=CONFIG.exec_mode != ExecMode.LAUNCH_MODULE)

LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)

CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.self_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG = CONFIG._replace(private_data_path=os.path.join(CONFIG.data_path, CONFIG.subproject_name))
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)

MODEL_REGISTRY = ModelRegistry(CONFIG.model_group_uri, download_nexus_url='http://nexus-slave:8081')

os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

# Hyperparameters

In [ ]:
# @launchit.disable
# @launchit.collect
@dataclass
class Hyperparameters:
    random_seed: int = None
    # dataset params
    db_fname: str = None
    # model params
    model_artifact_source: str = None

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

HP = Hyperparameters()
HP.random_seed = 42

# Dataset

## Configure

In [ ]:
# @launchit.disable
# @launchit.collect
HP.db_fname = 'dataset_5_128.db'
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

## get_db_con

In [ ]:
def get_db_con(hp=None):
    hp = LangUtils.coalesce(hp, HP)
    assert hp.db_fname, f'Uninitialized {hp.db_fname=}'
    return sqlite3.connect(f'file:{os.path.join(CONFIG.private_data_path, hp.db_fname)}?mode=ro', uri=True)

## DataComponents

In [ ]:
@dataclass
class DataComponents:
    meta: object = None
    images: dict = None
    test_images: dict = None
    image_labels: dict = None
    test_image_labels: dict = None
    vocab_token_ind_to_vocab_token: dict = None
    vocab_ind_to_vocab_token: dict = None
    vocab: object = None
    pos_token_ind_to_pos_token: dict = None
    df_samples: object = None
    df_test_samples: object = None
    pos_token_inds: object = None
    pos_tokens: object = None

    def get_image(self, image_ind, is_test):
        return (self.images, self.test_images)[is_test][image_ind]

    def get_image_label(self, image_ind, is_test):
        return (self.image_labels, self.test_image_labels)[is_test][image_ind]

DC = DataComponents()

## Load

In [ ]:
with get_db_con() as db_con:
    DC.meta = load_meta(db_con)
    image_and_labels = db_con.execute('SELECT image_ind, data, label FROM images').fetchall()
    DC.images = dict(map(lambda i: (i[0], np.load(BytesIO(i[1]))), image_and_labels))
    DC.image_labels = dict(map(lambda i: (i[0], i[2]), image_and_labels))
    image_and_labels = db_con.execute('SELECT image_ind, data, label FROM test_images').fetchall()
    DC.test_images = dict(map(lambda i: (i[0], np.load(BytesIO(i[1]))), image_and_labels))
    DC.test_image_labels = dict(map(lambda i: (i[0], i[2]), image_and_labels))
    DC.vocab_token_ind_to_vocab_token = {}
    DC.vocab_ind_to_vocab_token = {}
    DC.vocab = []

    for vt in load_vocab_tokens(db_con).itertuples():
        if vt.data is not None:
            vt = vt._replace(data=np.load(BytesIO(vt.data)))
            DC.vocab.append(vt.data.ravel())

        DC.vocab_token_ind_to_vocab_token[vt.Index] = vt
        DC.vocab_ind_to_vocab_token[vt.vocab_ind] = vt

    DC.vocab = np.ascontiguousarray(np.array(DC.vocab).T)

    DC.pos_token_ind_to_pos_token = dict(map(lambda t: (t.Index, t), load_pos_tokens(db_con).itertuples()))
    
    for attr_name, table_name, images_table_name in zip(
        ('df_samples', 'df_test_samples'), 
        ('noncausal_samples', 'test_noncausal_samples'), 
        ('images', 'test_images')
    ):
        setattr(DC, attr_name, pd.read_sql(f'SELECT s.image_ind, s.vocab_token_inds, s.pos_token_inds, s.p_matrix, i.label ' + 
                                           f'FROM {table_name} s, {images_table_name} i ' + 
                                           f'WHERE s.image_ind=i.image_ind', con=db_con))
        df = getattr(DC, attr_name)
        df.vocab_token_inds = df.vocab_token_inds.map(lambda i: np.array(list(map(int, i.split(',')))))
        df.pos_token_inds = df.pos_token_inds.map(lambda i: np.array(list(map(int, i.split(',')))))
        df.p_matrix = df.p_matrix.map(lambda m: np.load(BytesIO(m)))

    # Make sure all samples (both train and test) have the same positional tokens
    DC.pos_token_inds = None

    for df in (DC.df_samples, DC.df_test_samples):
        for sample in df.itertuples():
            if DC.pos_token_inds is None:
                DC.pos_token_inds = sample.pos_token_inds
            else:
                assert np.all(DC.pos_token_inds == sample.pos_token_inds), image_ind
        
        df.drop('pos_token_inds', axis=1, inplace=True)

    DC.pos_tokens = list(map(lambda i: DC.pos_token_ind_to_pos_token[i.item()], DC.pos_token_inds))   

## AugmentedDataset

In [ ]:
# li = 50
# lml = 3
# lt = 10
# l = li * lml * lt

# trace = []

# for i in range(li):
#     for j in range(lml):
#         for k in range(lt):
#             trace.append((i,j,k))
            
# for index in range(l):
#     slots_per_image_ind = lml * lt
#     image_ind = index // slots_per_image_ind
#     slots_per_masking_level_ind = lt
#     masking_level_ind = (index % slots_per_image_ind) // lt
#     transform_ind = (index % slots_per_image_ind) % lt
#     assert trace[index] == (image_ind, masking_level_ind, transform_ind)

In [ ]:
class AugmentedDataset(Dataset):
    def __init__(self, masking_levels, mask_len, transforms, images, image_labels):
        self.masking_levels = masking_levels
        self.mask_len = mask_len
        self.transforms = transforms
        self.images = images
        self.image_labels = image_labels

    def __len__(self):
        return len(self.images) * len(self.masking_levels) * len(self.transforms)

    Item = namedtuple('Item', 'image, label, mask, image_ind, masking_level, transform_ind')

    def __getitem__(self, index):
        slots_per_image_ind = len(self.masking_levels) * len(self.transforms)
        image_ind = index // slots_per_image_ind
        slots_per_masking_level_ind = len(self.transforms)
        masking_level_ind = (index % slots_per_image_ind) // len(self.transforms)
        transform_ind = (index % slots_per_image_ind) % len(self.transforms)
        return AugmentedDataset.Item(
            image=self.images[image_ind], 
            label=self.image_labels[image_ind], 
            mask=generate_masks(self.mask_len, [self.masking_levels[masking_level_ind]])[0], 
            image_ind=image_ind,
            masking_level=self.masking_levels[masking_level_ind],
            transform_ind=transform_ind,
        )

## AugmentedDatasetItemsCollator

In [ ]:
class AugmentedDatasetItemsCollator:
    def __init__(self, dc, transforms, p_matrix_n):
        self.dc = dc
        self.transforms = transforms
        self.p_matrix_n = p_matrix_n
        self.vocab = torch.tensor(dc.vocab)
        self.p_padding = np.zeros(dc.meta.system_tokens_count, dtype=np.float32)

    def __call__(self, items):
        images = []
        
        for item in items:
            image = torch.tensor(item.image)
            transform = self.transforms[item.transform_ind]
            image = tvtf.affine(einops.rearrange(image, 'h w -> 1 h w'), **transform)
            images.append(image)

        images = torch.vstack(images)
        
        patches = []
        
        for pt in self.dc.pos_tokens: 
            pt_patches = images[:,pt.i:pt.i2,pt.j:pt.j2] # [items,patch_size,patch_size]
            patches.append(pt_patches)
        
        patches = torch.vstack(patches) # [batch=|pos_tokens|*|items|,patch_size,patch_size]
        
        batch_loss_matrices = bp_batch_solo(einops.rearrange(patches, 'b h w -> b (h w)'), self.vocab, result_type='loss_matrix')
        # Here we have for for each image an assigned loss_matrix (and for each patch - an assigned loss vector/logits)
        batch_loss_matrices = einops.rearrange(batch_loss_matrices, '(p i) l -> i p l', i=len(items))
        batch_loss_matrices *= -1 # turn loss into somewhat that resambles goodness (the more the better)
        origin_vocab_token_inds = np.zeros((len(items), len(self.dc.pos_tokens)), dtype=np.int32)
        masked_vocab_token_inds = np.zeros((len(items), len(self.dc.pos_tokens)), dtype=np.int32)
        p_matrices = []
        
        for i, image_loss_matrix in enumerate(batch_loss_matrices): # per image cycle
            mask = items[i].mask
            p_matrix = []
            
            for j, logits in enumerate(image_loss_matrix): # per image's patch cycle
                top = torch.topk(logits, self.p_matrix_n) 
                ratios = -top.values / (top.values[0] + 1e-10) # epsilon for numerical stability
                probs = np.zeros_like(logits, dtype=np.float32)
                probs[top.indices] = torch.softmax(ratios, 0)
                p_matrix.append(np.r_[self.p_padding, probs])
                vocab_ind = top.indices[0].item()
                vt = self.dc.vocab_ind_to_vocab_token[vocab_ind]
                origin_vocab_token_inds[i,j] = vt.Index
                masked_vocab_token_inds[i,j] = LangUtils.when(mask[j], NoncausalTransformer.MASK_TOKEN_IND, vt.Index)
        
            p_matrix = np.array(p_matrix)
            p_matrices.append(p_matrix)

        return dict(
            image_inds=list(map(lambda i: i.image_ind, items)),
            masking_levels=list(map(lambda i: i.masking_level, items)),
            origin_vocab_token_inds=torch.tensor(np.array(origin_vocab_token_inds)), # [batch,seq_len]
            masked_vocab_token_inds=torch.tensor(np.array(masked_vocab_token_inds)), # [batch,seq_len]
            masks=torch.tensor(np.array(list(map(lambda i: i.mask, items)))), # [batch,seq_len]
            pred_targets=torch.tensor(np.array(p_matrices)), # [batch,seq_len,vocab_len]
            cls_targets=torch.tensor(np.array(list(map(lambda i: i.label, items)))), # [batch]
        )

# Model

## Configure

In [ ]:
# @launchit.disable
# @launchit.collect
HP.model_artifact_source = '15f_combo_noncausal_01:3'
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

## Create

In [ ]:
asp = hp_parse_artifact_source(HP.model_artifact_source)
mp = MODEL_REGISTRY.get_asset_content(asp.model_name, asp.model_version, 'json', 'model_params')
mp = json.loads(mp.decode())
cmp = ComboModel.Params(**mp)
LOG(f'Model params={dataclasses.asdict(cmp)}')
pt = MODEL_REGISTRY.get_asset_content(asp.model_name, asp.model_version, 'pt', 'model')
model = ComboModel(cmp)
model.load_state_dict(torch.load(BytesIO(pt), map_location=torch.device('cpu')))
model = model.to(CONFIG.cuda_device)
model.eval()
model.requires_grad_(False)
model

# Utils

## generate_masks

In [ ]:
def generate_masks(l, masking_steps):
    masks = np.zeros((len(masking_steps), l), dtype=bool)
    prev_masked_count = None
    free_inds = list(range(l))
    
    for i, masked_percentage in enumerate(masking_steps):
        masked_count = int(masked_percentage * l)
        assert 0 <= masked_count <= l, masked_count
        prev_masked_count = LangUtils.coalesce(prev_masked_count, 0)
        assert prev_masked_count <= masked_count, (prev_masked_count, masked_count, masked_percentage)
        items_to_mask = masked_count - prev_masked_count
        prev_masked_count = masked_count

        if items_to_mask == 0:
            continue

        assert items_to_mask <= len(free_inds), (len(free_inds), items_to_mask, masked_percentage)

        inds_for_masking = RNG.choice(free_inds, items_to_mask, replace=False)

        if i > 0:
            masks[i][:] = masks[i-1]
            
        masks[i,inds_for_masking] = True
        free_inds = list(set(free_inds) - set(inds_for_masking))

    return masks

In [ ]:
masking_steps = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
masks = generate_masks(25, masking_steps)
fig, axes = plt.subplots(1, len(masks))
fig.set_figwidth(18)

for mask, ax, mp in zip(masks, axes, masking_steps):
    ax.set_title(f'p={mp*100:.0f}% ({mask.sum()} / {len(mask)})', fontdict=dict(fontsize=10))
    ax.imshow(ArrayUtils.v2sm(mask))
    ax.set_axis_off()

## mask_vocab_token_inds

In [ ]:
def mask_vocab_token_inds(vocab_token_inds, masks, mask_value, device=None):
    origin_batch = vocab_token_inds
    assert masks.shape[1] == origin_batch.shape[1], (masks.shape[1], origin_batch.shape[1])
    masked_batches = []

    for mask in masks:
        masked_batch = origin_batch.copy()
        masked_batch[np.arange(len(masked_batch)),np.argwhere(mask)] = mask_value
        masked_batches.append(masked_batch)

    masked_batches = np.array(masked_batches)
    shape = einops.parse_shape(masked_batches, 'm b s')
    masked_batches = einops.rearrange(masked_batches, 'm b s -> (m b) s')
    return torch.tensor(masked_batches).to(device=LangUtils.coalesce(device, CONFIG.cuda_device)), shape

## visualize_model_output

In [ ]:
def visualize_model_output(model, df_samples, masking_levels, masks, is_test, device=None):
    batch = collate_samples_to_batch(df_samples, masks, mask_value=NoncausalTransformer.MASK_TOKEN_IND, device=device)
    batch_shape = einops.parse_shape(batch.vocab_token_inds, 'm b s')
    
    with eval_guard(model):
        with torch.no_grad(): 
            pred_logits, cls_logits = model(
                einops.rearrange(batch.vocab_token_inds, 'm b s -> (m b) s'), 
                torch.tensor(DC.pos_token_inds).to(device=device)
            )
            pred_logits = pred_logits.detach().cpu().numpy()
            pred_logits = einops.rearrange(pred_logits, '(m b) s l -> m b s l', b=batch_shape['b'])
            predictions = np.argmax(pred_logits, axis=-1)
            cls_logits = cls_logits.detach().cpu().numpy()
            cls_logits = einops.rearrange(cls_logits, '(m b) l -> m b l', b=batch_shape['b'])
            classes = np.argmax(cls_logits, axis=-1) 
    
    def generate_mosaic(samples_count, masking_levels_count):
        mosaic = []
        row = []
    
        for i in range(samples_count):
            row.extend([chr(ord('A') + i)] * 2)
    
        mosaic.append(row)
            
        for j in range(masking_levels_count):
            row = []
            
            for i in range(samples_count):
                prefix = chr(ord('A') + i)
                row.extend([f'{prefix}{j+1}1', f'{prefix}{j+1}2'])
            
            mosaic.append(row)
    
        return mosaic, ''.join(map(chr, ord('A') + np.arange(samples_count)))
    
    mosaic, major_letters = generate_mosaic(len(df_samples), len(masking_levels))
    fig, axd = plt.subplot_mosaic(mosaic, layout="constrained")
    fig.set_figwidth(16)
    fig.set_figheight(2 * len(mosaic))
    
    # Original image
    for sample_row, ax_name in zip(df_samples.itertuples(), major_letters):
        image = DC.get_image(sample_row.image_ind, is_test=is_test)
        axd[ax_name].set_title(f'{LangUtils.when(is_test, 'Test image', 'Image')} #{sample_row.image_ind}, {sample_row.label}')
        axd[ax_name].imshow(image)
        axd[ax_name].set_axis_off()
    
    # Masked image
    for i in range(len(batch.vocab_token_inds)):
        for j, ax_name_prefix in zip(range(len(batch.vocab_token_inds[i])), major_letters):
            assert len(batch.vocab_token_inds[i,j]) == len(DC.pos_token_inds)
            image_hat = np.zeros((DC.meta.image_size, DC.meta.image_size))
            ax = axd[f'{ax_name_prefix}{i+1}1']
            
            for vocab_token_ind, pos_token_ind in zip(batch.vocab_token_inds[i,j], DC.pos_token_inds):
                vt = DC.vocab_token_ind_to_vocab_token[vocab_token_ind.item()]
                pt = DC.pos_token_ind_to_pos_token[pos_token_ind.item()]
    
                if vocab_token_ind.item() == NoncausalTransformer.MASK_TOKEN_IND:
                    rect = pltpatches.Rectangle((pt.j-0.5, pt.i-0.5), DC.meta.patch_size, DC.meta.patch_size, linewidth=1, edgecolor='w', facecolor='none')
                    ax.add_patch(rect)
                else:
                    image_hat[pt.i:pt.i2,pt.j:pt.j2] = ArrayUtils.v2sm(vt.data)
    
            ax.set_title(f'Masking level={masking_levels[i]*100:.0f}%', fontdict=dict(fontsize=10))
            ax.imshow(image_hat)
            ax.set_axis_off()
    
    # Maked image with filled in items
    for i in range(len(predictions)):
        for j, ax_name_prefix in zip(range(len(predictions[i])), major_letters):
            assert len(predictions[i,j]) == len(DC.pos_token_inds)
            assert len(predictions[i,j]) == len(batch.vocab_token_inds[i,j])
            assert len(predictions[i,j]) == len(batch.masks[i,j])
            image_hat = np.zeros((DC.meta.image_size, DC.meta.image_size))
            ax = axd[f'{ax_name_prefix}{i+1}2']
            
            for k in range(len(predictions[i,j])):
                pt = DC.pos_token_ind_to_pos_token[DC.pos_token_inds[k]]
                mask_value = batch.masks[i,j,k].item()
                
                if mask_value:
                    # masked value - get from model's output (prediction)
                    vt = DC.vocab_token_ind_to_vocab_token[predictions[i,j,k]]
                    color = LangUtils.when(batch.targets[i,j,k,vt.Index] > 0.01, 'g', 'r')
                    lw = LangUtils.when(batch.targets[i,j,k,vt.Index] > 0.01, 2, 1)
                    rect = pltpatches.Rectangle((pt.j-0.5, pt.i-0.5), DC.meta.patch_size, DC.meta.patch_size, linewidth=lw, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                else:
                    # unmasked value - get from model's input
                    vt = DC.vocab_token_ind_to_vocab_token[batch.vocab_token_inds[i,j,k].item()]
                    
    
                if vt.data is not None:
                    image_hat[pt.i:pt.i2,pt.j:pt.j2] = ArrayUtils.v2sm(vt.data)
    
            ax.set_title(f'Masking level={masking_levels[i]*100:.0f}%, label={classes[i,j]}', fontdict=dict(fontsize=10))
            ax.imshow(image_hat)
            ax.set_axis_off()

    return fig

# Explore

## get_default_transforms

In [ ]:
def get_default_transforms():
    return [
        dict(translate=(0, 0), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(2, 0), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(-2, 0), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(0, 2), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(0, -2), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(2, 2), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(2, -2), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(-2, -2), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(-2, 2), angle=0, scale=1.0, shear=[0, 0]),
        dict(translate=(0, 0), angle=-15, scale=1.0, shear=[0, 0]),
        dict(translate=(0, 0), angle=+15, scale=1.0, shear=[0, 0]),
    ]

## visualize_transforms_and_masking_levels

In [ ]:
def visualize_transforms_and_masking_levels(transforms, masking_levels, is_test):
    only_pos_token_inds = torch.tensor(DC.pos_token_inds).to(CONFIG.cuda_device)
    masks = generate_masks(len(DC.pos_token_inds), masking_levels)
    image_ind = RNG.choice(list([DC.images, DC.test_images][is_test].keys()))
    image = einops.rearrange(torch.tensor(DC.get_image(image_ind, is_test=is_test)), 'h w -> 1 h w')
    image_label = DC.get_image_label(image_ind, is_test)
    label_match_counter = Counter()
    
    fig, axes = plt.subplots(len(transforms), len(masking_levels) * 3, layout='constrained')
    fig.set_figwidth(18)
    fig.set_figheight(2 * len(axes))
    
    for i, transform in enumerate(transforms):
        full_kwargs = dict(transform)
        full_kwargs.update(dict(scale=1.0, shear=[0, 0]))
        image_hat = tvtf.affine(image, **full_kwargs)[0].numpy()
        
        for j, (masking_level, mask) in enumerate(zip(masking_levels, masks)):
            patches = []
    
            for pti in DC.pos_token_inds:
                pt = DC.pos_token_ind_to_pos_token[pti.item()]
                patches.append(image_hat[pt.i:pt.i2,pt.j:pt.j2].ravel())
    
            patches = np.array(patches)
            vocab_inds = np.argmax(bp_batch_solo(patches, DC.vocab), axis=-1)
            
            ax_origin = axes[i,j*3+0]
            ax_dirty = axes[i,j*3+1]
            ax_restored = axes[i,j*3+2]
    
            image_origin = np.zeros_like(image_hat)
            image_dirty = np.zeros_like(image_hat)
            image_restored = np.zeros_like(image_hat)
            
            for pti, vi, mask_item in zip(DC.pos_token_inds, vocab_inds, mask):
                pt = DC.pos_token_ind_to_pos_token[pti.item()]
                vt = DC.vocab_ind_to_vocab_token[vi]
                image_origin[pt.i:pt.i2,pt.j:pt.j2] = ArrayUtils.v2sm(vt.data)
    
                if not mask_item:
                    image_dirty[pt.i:pt.i2,pt.j:pt.j2] = ArrayUtils.v2sm(vt.data)
                else:
                    rect = pltpatches.Rectangle((pt.j-0.5, pt.i-0.5), DC.meta.patch_size, DC.meta.patch_size, linewidth=1, edgecolor='w', facecolor='none')
                    ax_dirty.add_patch(rect)

            desc = f'off=[{transform["translate"][0]:+},{transform["translate"][1]:+}], r={transform["angle"]}'
            ax_origin.set_title(desc, fontdict=dict(fontsize=10))
            ax_origin.imshow(image_origin)
    
            ax_dirty.set_title(f'ml={masking_level*100:.0f}%', fontdict=dict(fontsize=10))
            ax_dirty.imshow(image_dirty)
    
            vocab_token_inds = []
    
            for vi in vocab_inds:
                vt = DC.vocab_ind_to_vocab_token[vi]
                vocab_token_inds.append(vt.Index)
    
            vocab_token_inds = np.array(vocab_token_inds)
            assert len(vocab_token_inds) == len(DC.pos_token_inds)
            masked_vocab_token_inds, shape = mask_vocab_token_inds(
                einops.rearrange(vocab_token_inds, 's -> 1 s'), 
                einops.rearrange(mask, 's -> 1 s'), 
                NoncausalTransformer.MASK_TOKEN_IND)
            pred_logits, cls_logits = model(masked_vocab_token_inds, only_pos_token_inds)
            pred_logits = einops.rearrange(pred_logits, '1 s l -> s l')
            pred_vocab_token_inds = torch.argmax(pred_logits, axis=-1) 
    
            for vti, pvti, pti, mv in zip(vocab_token_inds, pred_vocab_token_inds, DC.pos_token_inds, mask):
                pt = DC.pos_token_ind_to_pos_token[pti.item()]
    
                if mv:
                    # masked value - get from model's output (prediction)
                    vt = DC.vocab_token_ind_to_vocab_token[pvti.item()]
                    rect = pltpatches.Rectangle((pt.j-0.5, pt.i-0.5), DC.meta.patch_size, DC.meta.patch_size, linewidth=1, edgecolor='w', facecolor='none')
                    # ax_restored.add_patch(rect)
                else:
                    # unmasked value - get from model's input
                    vt = DC.vocab_token_ind_to_vocab_token[vti.item()]
    
                if vt.data is not None:
                    image_restored[pt.i:pt.i2,pt.j:pt.j2] = ArrayUtils.v2sm(vt.data)

            label = torch.argmax(cls_logits, axis=-1).item()
            is_label_match = image_label == label
            label_match_counter[is_label_match] += 1
            ax_restored.set_title(f'{label=}', fontdict=dict(fontsize=10), color=LangUtils.when(is_label_match, 'g', 'r'))
            ax_restored.imshow(image_restored)
    
            for ax in ax_origin, ax_dirty, ax_restored:
                ax.set_axis_off()

    match_rate_true = label_match_counter[True]
    match_rate_total = label_match_counter.total()
    match_rate = match_rate_true / match_rate_total
    fig.suptitle(f'{LangUtils.when(is_test, "Test image", "Image")} #{image_ind}, match rate={match_rate*100:.0f}% {match_rate_true}/{match_rate_total}')
    return fig

### is_test=False

In [ ]:
fig = visualize_transforms_and_masking_levels(get_default_transforms(), masking_levels=[0.0, 0.2, 0.4], is_test=False)

### is_test=True

In [ ]:
fig = visualize_transforms_and_masking_levels(get_default_transforms(), masking_levels=[0.0, 0.2, 0.4], is_test=True)

## visualize_transforms_and_images

In [ ]:
def visualize_transforms_and_images(transforms, image_inds, is_test):
    only_pos_token_inds = torch.tensor(DC.pos_token_inds).to(CONFIG.cuda_device)
    masks = generate_masks(len(DC.pos_token_inds), [0])
    fig, axes = plt.subplots(len(transforms), len(image_inds), layout='constrained')
    fig.set_figwidth(18)
    fig.set_figheight(2 * len(axes))
    
    label_match_counter = Counter()
    
    for j, image_ind in enumerate(image_inds):
        image = einops.rearrange(torch.tensor(DC.get_image(image_ind, is_test=is_test)), 'h w -> 1 h w')
        image_label = DC.get_image_label(image_ind, is_test)
        
        for i, transform in enumerate(transforms):
            full_kwargs = dict(transform)
            full_kwargs.update(dict(scale=1.0, shear=[0, 0]))
            image_hat = tvtf.affine(image, **full_kwargs)[0].numpy()
            patches = []
    
            for pti in DC.pos_token_inds:
                pt = DC.pos_token_ind_to_pos_token[pti.item()]
                patches.append(image_hat[pt.i:pt.i2,pt.j:pt.j2].ravel())
    
            patches = np.array(patches)
            vocab_inds = np.argmax(bp_batch_solo(patches, DC.vocab), axis=-1)
            
            ax_origin = axes[i,j]
    
            image_origin = np.zeros_like(image_hat)
            
            for pti, vi in zip(DC.pos_token_inds, vocab_inds):
                pt = DC.pos_token_ind_to_pos_token[pti.item()]
                vt = DC.vocab_ind_to_vocab_token[vi]
                image_origin[pt.i:pt.i2,pt.j:pt.j2] = ArrayUtils.v2sm(vt.data)
    
            ax_origin.imshow(image_origin)
    
            vocab_token_inds = []
    
            for vi in vocab_inds:
                vt = DC.vocab_ind_to_vocab_token[vi]
                vocab_token_inds.append(vt.Index)
    
            vocab_token_inds = np.array(vocab_token_inds)
            assert len(vocab_token_inds) == len(DC.pos_token_inds)
            masked_vocab_token_inds, shape = mask_vocab_token_inds(
                einops.rearrange(vocab_token_inds, 's -> 1 s'), 
                einops.rearrange(masks[0], 's -> 1 s'), 
                NoncausalTransformer.MASK_TOKEN_IND)
            _, cls_logits = model(masked_vocab_token_inds, only_pos_token_inds)
    
            label = torch.argmax(cls_logits, axis=-1).item()
            is_label_match = image_label == label
            label_match_counter[is_label_match] += 1
            desc = f'off=[{transform["translate"][0]:+},{transform["translate"][1]:+}], r={transform["angle"]}'
            ax_origin.set_title(f'{LangUtils.when(is_test, "Test image", "Image")} #{image_ind},\n' + 
                                f'{desc}, {label=}', fontdict=dict(fontsize=10), color=LangUtils.when(is_label_match, 'g', 'r'))
            ax_origin.set_axis_off()
        
    match_rate_true = label_match_counter[True]
    match_rate_total = label_match_counter.total()
    match_rate = match_rate_true / match_rate_total
    fig.suptitle(f'Match rate={match_rate*100:.0f}% {match_rate_true}/{match_rate_total}')
    return fig

### is_test=False

In [ ]:
image_inds = RNG.choice(list(DC.images.keys()), 9)
visualize_transforms_and_images(get_default_transforms(), image_inds, is_test=False)

### is_test=True

In [ ]:
image_inds = RNG.choice(list(DC.test_images.keys()), 9)
visualize_transforms_and_images(get_default_transforms(), image_inds, is_test=True)

## batch_accuracy

In [ ]:
def get_batch_accuracy(masking_levels, transforms, batch_size, is_test):
    only_pos_token_inds = torch.tensor(DC.pos_token_inds).to(CONFIG.cuda_device)
    dataset = AugmentedDataset(masking_levels, len(DC.pos_token_inds), transforms, (DC.images, DC.test_images)[is_test], (DC.image_labels, DC.test_image_labels)[is_test])
    collator = AugmentedDatasetItemsCollator(DC, transforms, 3)
    data_loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collator)
    batch = next(iter(data_loader))
    vocab_token_inds = batch['masked_vocab_token_inds'].to(CONFIG.cuda_device)
    cls_targets = batch['cls_targets'].to(CONFIG.cuda_device)
    _, cls_logits = model(vocab_token_inds, only_pos_token_inds)
    f = RecursiveAverageFilter()
    match_map = torch.argmax(cls_logits, axis=1) == cls_targets
    f(1, batch_size=(match_map == True).sum().item())
    f(0, batch_size=(match_map == False).sum().item())
    return f

### is_test=False

In [ ]:
get_batch_accuracy([0], get_default_transforms(), 1000, False)

### is_test=True

In [ ]:
get_batch_accuracy([0], get_default_transforms(), 1000, True)

# LaunchIt!

In [ ]:
# @launchit.disable
launchit_t0 = time.time()

In [ ]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.self_name}'))
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=model_version, expandvars=expandvars)
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')